01_data_preparation.ipynb

1. Import libraries

2. Dataset path check

3. Read JSON files

4. Create Documents

5. Verify Documents
   Total Documents: 102

6. Save documents.pkl  ✅

In [12]:
from pathlib import Path
import json
import pickle

from langchain_core.documents import Document

# ============================
# Dataset Path
# ============================

DATA_PATH = Path("data/egypt_landmarks_kb")

print("Dataset Exists:", DATA_PATH.exists())

documents = []

# ============================
# Read Dataset
# ============================

for landmark_folder in DATA_PATH.iterdir():

    if not landmark_folder.is_dir():
        continue

    info_file = landmark_folder / "info.json"
    faq_file = landmark_folder / "faq.json"

    if not info_file.exists():
        continue

    with open(info_file, "r", encoding="utf-8") as f:
        info = json.load(f)

    text = ""

    text += f"Landmark Name: {info.get('name','')}\n"
    text += f"Category: {info.get('category','')}\n\n"

    text += f"Location:\n{info.get('location','')}\n\n"

    text += f"Historical Description:\n{info.get('historical_description','')}\n\n"

    # ============================
    # Construction Details
    # ============================

    construction = info.get("construction_details", {})

    if isinstance(construction, dict):

        text += "Construction Details:\n"

        for key, value in construction.items():

            key = key.replace("_", " ").title()

            text += f"{key}: {value}\n"

        text += "\n"

    # ============================
    # FAQ
    # ============================

    if faq_file.exists():

        with open(faq_file, "r", encoding="utf-8") as f:
            faq = json.load(f)

        if isinstance(faq, list):
            faq_items = faq

        elif isinstance(faq, dict):
            faq_items = faq.get("faqs", [])

        else:
            faq_items = []

        text += "Frequently Asked Questions\n\n"

        for item in faq_items:

            question = item.get("question", "")
            answer = item.get("answer", "")

            text += f"Question: {question}\n"
            text += f"Answer: {answer}\n\n"

    documents.append(

        Document(

            page_content=text,

            metadata={
                "landmark_id": landmark_folder.name
            }

        )

    )

# ============================
# Results
# ============================

print("=" * 80)
print("Total Documents:", len(documents))
print("=" * 80)

print(documents[0].page_content[:4000])

# ============================
# Save
# ============================

with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

print("\nDocuments saved successfully!")

Dataset Exists: True
Total Documents: 6
Landmark Name: Abu Simbel Temples
Category: Rock-Cut Ancient Temple Complex / UNESCO World Heritage Site

Location:
{'city': 'Abu Simbel', 'governorate': 'Aswan', 'country': 'Egypt', 'region': 'Southern Upper Egypt, western shore of Lake Nasser', 'coordinates': {'latitude': 22.3372, 'longitude': 31.6258}}

Historical Description:
Abu Simbel consists of two massive rock-cut temples carved into a sandstone cliff during the reign of Ramesses II, one of ancient Egypt's most powerful pharaohs. The larger Great Temple, fronted by four colossal 20-meter statues of Ramesses II, was dedicated to himself and the gods Ra-Horakhty, Amun, and Ptah, primarily to project Egyptian power toward Nubia and commemorate his reign, including his account of the Battle of Kadesh. The smaller temple, dedicated to the goddess Hathor and Ramesses II's chief wife Queen Nefertari, is one of the few ancient Egyptian temples where a queen is depicted at the same scale as the k